# Voice-Model: GPU run (encoder eval, Stage A, Stage B, evaluation)

Run the cells top to bottom on the GPU machine, from a clone of the repo. Training cells resume from `last.pt` if the kernel dies, so you can rerun them.

Before you start, copy `encoder_best_step12000.pt` from Google Drive into `ARTIFACTS` (see the checkpoint cell).

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path.cwd() if (Path.cwd() / 'stage_b.py').exists() else Path.cwd().parent
DATA_DIR = REPO_DIR
ARTIFACTS = REPO_DIR / 'artifacts'
LIBRISPEECH_ROOT = None
STAGE_A_STEPS = None
STAGE_B_STEPS = None
EXTRA_ARGS = []
INSTALL_DEPS = True
DOWNLOAD_LJSPEECH = True

if not (REPO_DIR / 'stage_b.py').exists():
    raise FileNotFoundError(f'stage_b.py not found in {REPO_DIR}; open this notebook from the repo root')
ARTIFACTS.mkdir(parents=True, exist_ok=True)
os.environ['VM_DRIVE_DIR'] = str(ARTIFACTS)
os.environ['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('repo     :', REPO_DIR)
print('data     :', DATA_DIR)
print('artifacts:', ARTIFACTS)

In [ ]:
import shutil
import subprocess


def run(script, *args):
    cmd = [sys.executable, '-u', str(REPO_DIR / script), *map(str, args)]
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=DATA_DIR, env=os.environ.copy(), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'{script} exited with code {proc.returncode}')


if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
if INSTALL_DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.10.2', 'peft==0.19.1',
                    'accelerate', 'speechbrain>=1.0', 'soundfile', 'librosa', 'scipy', 'scikit-learn',
                    'pandas'], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available; this notebook needs a GPU')
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM {props.total_memory / 1e9:.1f} GB | torch {torch.__version__}')
if props.total_memory < 15e9:
    print('warning: under 16 GB VRAM; lower the batch size by adding --batch_size 2 to EXTRA_ARGS')

## Data

In [ ]:
import tarfile
import urllib.request

lj = DATA_DIR / 'LJSpeech-1.1'
if not (lj / 'metadata.csv').exists():
    if not DOWNLOAD_LJSPEECH:
        raise FileNotFoundError(f'{lj} missing and DOWNLOAD_LJSPEECH is False')
    archive = DATA_DIR / 'LJSpeech-1.1.tar.bz2'
    if not archive.exists():
        print('downloading LJSpeech (2.6 GB)...')
        urllib.request.urlretrieve('https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2', archive)
    with tarfile.open(archive) as t:
        t.extractall(DATA_DIR)
n_wavs = len(list((lj / 'wavs').glob('*.wav')))
print(f'LJSpeech wavs: {n_wavs}')
if n_wavs < 13000:
    print('warning: expected 13,100 wavs; the extraction may be incomplete')

## Splits

The encoder was trained on the Colab `train.csv`/`val.csv`. Copy those two files into `DATA_DIR` if you have them. Otherwise they are regenerated with the same seed, and a warning is printed if they differ from existing files.

In [ ]:
os.chdir(DATA_DIR)
import common as C

C.ensure_splits()
for name in (C.TRAIN_CSV, C.VAL_CSV, C.TEST_CSV):
    print(name, len(C.read_split(name)))

## Encoder checkpoint

In [ ]:
ckpt_path = ARTIFACTS / 'encoder_best_step12000.pt'
if not ckpt_path.exists():
    raise FileNotFoundError(f'copy encoder_best_step12000.pt from Google Drive (MyDrive) to {ckpt_path}')
ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
missing = {'encoder_state', 'decoder_state', 'ctc_head_state'} - set(ck)
if missing:
    raise RuntimeError(f'{ckpt_path} is missing {sorted(missing)}; it is not the full encoder checkpoint')
print(f'encoder checkpoint OK: step {ck.get("step")}, logged val loss {ck.get("val_loss")}')
del ck

## 1. Encoder metrics (Table I)

In [ ]:
run('evaluate.py', 'encoder', *EXTRA_ARGS)

## 2. Stage A training

In [ ]:
args = [*EXTRA_ARGS]
if STAGE_A_STEPS:
    args += ['--max_steps', STAGE_A_STEPS]
if LIBRISPEECH_ROOT:
    args += ['--librispeech', LIBRISPEECH_ROOT]
run('stage_a.py', *args)

In [ ]:
run('evaluate.py', 'stage_a', *EXTRA_ARGS)

## 3. Stage B training

In [ ]:
args = [*EXTRA_ARGS]
if STAGE_B_STEPS:
    args += ['--max_steps', STAGE_B_STEPS]
run('stage_b.py', 'train', *args)

## 4. Stage B evaluation (test split)

In [ ]:
run('evaluate.py', 'stage_b', *EXTRA_ARGS)

## 5. Results

In [ ]:
import json

results_dir = REPO_DIR / 'results'
results_dir.mkdir(exist_ok=True)
sources = {
    'encoder_metrics.json': ARTIFACTS / 'eval' / 'encoder_metrics.json',
    'stage_a_control.json': ARTIFACTS / 'eval' / 'stage_a_control.json',
    'stage_a_summary.json': ARTIFACTS / 'stage_a' / 'summary.json',
    'stage_a_metrics.csv': ARTIFACTS / 'stage_a' / 'metrics.csv',
    'stage_b_summary.json': ARTIFACTS / 'stage_b' / 'summary.json',
    'stage_b_metrics.csv': ARTIFACTS / 'stage_b' / 'metrics.csv',
    'stage_b_metrics.json': ARTIFACTS / 'eval' / 'stage_b_metrics.json',
    'stage_b_per_utterance.csv': ARTIFACTS / 'eval' / 'stage_b_per_utterance.csv',
}
for name, src in sources.items():
    if src.exists():
        shutil.copy2(src, results_dir / name)
        print('copied', name)
    else:
        print('missing', src)


def show(path, keys):
    if not path.exists():
        return
    data = json.loads(path.read_text())
    print(f'\n{path.name}')
    for k in keys:
        print(f'  {k}: {json.dumps(data.get(k))}')


show(results_dir / 'encoder_metrics.json', ['losses_as_trained', 'ctc_greedy_cer', 'geometry_padded_meanpool'])
show(results_dir / 'stage_a_summary.json', ['best_step', 'best_val_loss'])
show(results_dir / 'stage_a_control.json', ['text_ce'])
show(results_dir / 'stage_b_summary.json', ['best_step', 'best_val'])
show(results_dir / 'stage_b_metrics.json', ['judge_wer', 'mcd_dtw_db', 'llm_transcript', 'teacher_forced'])

Commit `results/` so every number in the paper traces to a file. Audio samples are in `artifacts/eval/stage_b_samples/`.